## ⚙️ 0. Setup — imports et clés

In [1]:
import requests, json, os, tempfile
import pandas as pd
from datetime import date, timedelta, datetime
from pprint import pprint

# ──────────────────────────────────────────────────────────
# 🔑  CLÉS API  —  remplir ici avant de lancer le notebook
# ──────────────────────────────────────────────────────────

# 1. DataTourisme (clé permanente)
DT_API_KEY = "82a69518-beaa-46b4-bff4-12cbcb1a057a"

print("✅ Setup OK")

✅ Setup OK


---
# 🏰 PARTIE 1 — DataTourisme
**Base URL :** `https://api.datatourisme.fr/v1/`
**+530 000** points d'intérêt touristiques en France
**Auth :** header `X-API-Key`

### 1.1 — Test de connexion : premiers POIs

In [2]:
import requests, json

DT_BASE    = "https://api.datatourisme.fr/v1"
DT_API_KEY = "82a69518-beaa-46b4-bff4-12cbcb1a057a"
DT_HEADERS = {"X-API-Key": DT_API_KEY, "Accept": "application/json"}

resp = requests.get(f"{DT_BASE}/catalog",
                    headers=DT_HEADERS,
                    params={"lang": "fr", "page_size": 3})

data = resp.json()

# Imprimir la estructura CRUDA del primer POI
print("=== POI 0 — JSON COMPLETO ===")
print(json.dumps(data["objects"][0], indent=2, ensure_ascii=False))

=== POI 0 — JSON COMPLETO ===
{
  "hasBeenCreatedBy": {
    "legalName": "Haute Maurienne Vanoise Tourisme",
    "identifier": "582",
    "homepage": [
      "https://www.haute-maurienne-vanoise.com/"
    ]
  },
  "isLocatedAt": [
    {
      "geo": {
        "elevation": 2002,
        "latitude": 45.233402,
        "longitude": 6.722158
      },
      "address": [
        {
          "hasAddressCity": {
            "insee": "73026",
            "label": {
              "@fr": "Avrieux"
            }
          },
          "postalCode": "73500",
          "addressLocality": "Avrieux"
        }
      ]
    }
  ],
  "lastUpdate": "2026-02-03",
  "hasDescription": [
    {
      "shortDescription": {
        "@fr": "La télégraphie Chappe, ancêtre de nos télécommunications modernes, a vu le jour à la fin du 18ème siècle. Le poste de Courberon, implanté à 2002 mètres sur la commune d’Avrieux, faisait partie d’une ligne de communication allant de Paris à Venise."
      }
    }
  ],
  "label":

In [3]:
import requests, json, pandas as pd

DT_BASE    = "https://api.datatourisme.fr/v1"
DT_API_KEY = "82a69518-beaa-46b4-bff4-12cbcb1a057a"
DT_HEADERS = {"X-API-Key": DT_API_KEY, "Accept": "application/json"}

# ── Fonctions d'extraction basées sur la vraie structure de l'API ──

def extraire_nom(poi):
    label = poi.get("label", {})
    return label.get("@fr") or label.get("@en") or "?"

def extraire_geo(poi):
    locs = poi.get("isLocatedAt", [])
    if not locs:
        return None, None
    geo = locs[0].get("geo", {})
    return geo.get("latitude"), geo.get("longitude")

def extraire_ville(poi):
    locs = poi.get("isLocatedAt", [])
    if not locs:
        return "?"
    addresses = locs[0].get("address", [])
    if not addresses:
        return locs[0].get("address", {}).get("addressLocality", "?") if isinstance(locs[0].get("address"), dict) else "?"
    addr = addresses[0] if isinstance(addresses, list) else addresses
    # addressLocality est le plus fiable
    ville = addr.get("addressLocality") or addr.get("postalCode", "?")
    return ville

def extraire_description(poi):
    descs = poi.get("hasDescription", [])
    if not descs:
        return ""
    short = descs[0].get("shortDescription", {})
    return short.get("@fr") or short.get("@en") or ""

def extraire_photo(poi):
    imgs = poi.get("hasMainRepresentation", [])
    if not imgs:
        return None
    ressources = imgs[0].get("ebucore:hasRelatedResource", [])
    if not ressources:
        return None
    return ressources[0].get("ebucore:locator")

def extraire_contact(poi):
    contacts = poi.get("hasContact", [])
    if not contacts:
        return {}
    c = contacts[0]
    return {
        "tel":  (c.get("telephone") or [None])[0],
        "web":  (c.get("homepage")  or [None])[0],
        "mail": (c.get("email")     or [None])[0],
    }

def simplificar_poi(p):
    lat, lon = extraire_geo(p)
    return {
        "uuid":        p.get("uuid"),
        "nom":         extraire_nom(p),
        "type":        p.get("type", []),
        "lat":         float(lat or 0),
        "lon":         float(lon or 0),
        "ville":       extraire_ville(p),
        "description": extraire_description(p),
        "photo":       extraire_photo(p),
        "contact":     extraire_contact(p),
    }

# ── Test 1 : premiers POIs ──
resp = requests.get(f"{DT_BASE}/catalog",
                    headers=DT_HEADERS,
                    params={"lang": "fr", "page_size": 5})
data = resp.json()
print(f"✅ {data['meta']['total']:,} POIs au total\n")
for p in data["objects"]:
    poi = simplificar_poi(p)
    print(f"  🏛️  {poi['nom']:<50}  {poi['ville']:<20}  ({poi['lat']:.4f}, {poi['lon']:.4f})")

# ── Test 2 : recherche géospatiale ──
print("\n\n📍 POIs à 15 km de Montpellier :")
resp2 = requests.get(f"{DT_BASE}/catalog",
                     headers=DT_HEADERS,
                     params={"lang": "fr", "geo_distance": "43.6119,3.8772,15km", "page_size": 10})
data2 = resp2.json()
print(f"   Total : {data2['meta']['total']}\n")
pois = [simplificar_poi(p) for p in data2["objects"]]
for poi in pois:
    print(f"  📌 {poi['nom']:<50}  {poi['ville']:<20}  ({poi['lat']:.4f}, {poi['lon']:.4f})")

# ── DataFrame ──
print("\n")
pd.DataFrame(pois)[["nom", "ville", "lat", "lon", "description"]].head(10)

✅ 518,083 POIs au total

  🏛️  Télégraphe Chappe de Courberon                      Avrieux               (45.2334, 6.7222)
  🏛️  La roulotte du Château de Lescaut                   Montignac-de-Lauzun   (44.5702, 0.4474)
  🏛️  Camping du Dourdou                                  Brusque               (43.7609, 2.9526)
  🏛️  LA GUINGUETTE                                       Argens-Minervois      (43.2408, 2.7656)
  🏛️  Maison Jean HUTTARD                                 Zellenberg            (48.1715, 7.3205)


📍 POIs à 15 km de Montpellier :
   Total : 2257

  📌 EXIL PHOTO                                          Palavas-les-Flots     (43.5277, 3.9291)
  📌 PLAGE PRIVEE - ZENITH PLAGE                         Palavas-les-Flots     (43.5226, 3.9213)
  📌 POINT D'EAU POTABLE                                 Pignan                (43.5819, 3.7617)
  📌 CANDLELIGHT: JEAN-JACQUES GOLDMAN                   Montpellier           (43.6128, 3.8737)
  📌 DELICES DE TUNIS                              

,nom,ville,lat,lon,description
0,EXIL PHOTO,Palavas-les-Flots,43.527670,3.929090,"L’été , nous vous proposons de réaliser de bea..."
1,PLAGE PRIVEE - ZENITH PLAGE,Palavas-les-Flots,43.522600,3.921298,Paillote chaleureuse située Rive droite à Pala...
2,POINT D'EAU POTABLE,Pignan,43.581903,3.761729,
3,CANDLELIGHT: JEAN-JACQUES GOLDMAN,Montpellier,43.612770,3.873690,"Candlelight, c'est une expérience musicale mag..."
4,DELICES DE TUNIS,Palavas-les-Flots,43.527514,3.932091,
5,L'AMBASSADOR,Montpellier,43.609084,3.893991,"Restaurant de l’hôtel. Au cœur d’Antigone, pr..."
6,KWM KITESCHOOL CLUB,Villeneuve-lès-Maguelone,43.524701,3.855351,"Club/école de kite, KWM KITESCHOOL vous enseig..."
7,LE PATIO 34 - TERRASSE SUR PISCINE,Lavérune,43.579925,3.811763,
8,BORNE DE RECHARGE FRESHMILE,Mauguio,43.638738,3.969848,
9,CONCERT DU 14 JUILLET,Montpellier,43.598541,3.896917,Dans le cadre du Festival Radio France Occitan...


o---
# 🌦️ PARTIE 2 — Open-meteo
**Modèle numérique atmosphérique haute résolution** (0.01°, ~1 km)
**Couverture :** France métropolitaine | **Horizon :** 42h
**Auth :** Token Bearer  →  générer sur https://portail-api.meteofrance.fr

*texte en italique*### 2.1 — charger l'api


In [4]:
!pip install openmeteo-requests requests-cache retry-requests numpy pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.7/208.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 723.5/723.5 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.4/399.4 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.8 MB/s eta 0:00:00


In [5]:
import openmeteo_requests
import requests_cache
from retry_requests import retry
import pandas as pd

cache_session   = requests_cache.CachedSession('.cache', expire_after=600)
retry_session   = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo       = openmeteo_requests.Client(session=retry_session)

print("✅ Open-Meteo client prêt")

✅ Open-Meteo client prêt


In [6]:
# Codes météo WMO → description lisible
WMO_CODES = {
    0: "☀️ Ciel dégagé", 1: "🌤️ Peu nuageux", 2: "⛅ Partiellement nuageux",
    3: "☁️ Couvert", 45: "🌫️ Brouillard", 48: "🌫️ Givre",
    51: "🌦️ Bruine légère", 53: "🌦️ Bruine", 55: "🌧️ Bruine forte",
    61: "🌧️ Pluie légère", 63: "🌧️ Pluie", 65: "🌧️ Pluie forte",
    71: "🌨️ Neige légère", 73: "🌨️ Neige", 75: "❄️ Neige forte",
    80: "🌦️ Averses légères", 81: "🌧️ Averses", 82: "⛈️ Averses fortes",
    95: "⛈️ Orage", 96: "⛈️ Orage avec grêle", 99: "⛈️ Orage violent"
}

def get_meteo_complet(lat, lon):
    """
    Retourne la météo actuelle + prévisions 3 jours pour un point lat/lon.
    Pas de token nécessaire.
    """
    params = {
        "latitude":   lat,
        "longitude":  lon,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m",
            "precipitation",
            "weather_code",
            "apparent_temperature",
        ],
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "weather_code",
            "wind_speed_10m_max",
        ],
        "timezone":       "Europe/Paris",
        "forecast_days":  3,
        "past_days":      0,
    }

    responses = openmeteo.weather_api("https://api.open-meteo.com/v1/forecast", params=params)
    r = responses[0]

    # ── Données horaires → trouver l'heure actuelle ──
    hourly  = r.Hourly()
    dates   = pd.date_range(
        start=pd.to_datetime(hourly.Time(),    unit="s", utc=True).tz_convert("Europe/Paris"),
        end  =pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True).tz_convert("Europe/Paris"),
        freq =pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    )
    df_hourly = pd.DataFrame({
        "date":               dates,
        "temperature_2m":     hourly.Variables(0).ValuesAsNumpy(),
        "humidite_pct":       hourly.Variables(1).ValuesAsNumpy(),
        "vent_kmh":           hourly.Variables(2).ValuesAsNumpy(),
        "precipitation_mm":   hourly.Variables(3).ValuesAsNumpy(),
        "weather_code":       hourly.Variables(4).ValuesAsNumpy(),
        "temperature_ressentie": hourly.Variables(5).ValuesAsNumpy(),
    })

    # Heure actuelle (Paris)
    now     = pd.Timestamp.now(tz="Europe/Paris").floor("h")
    current = df_hourly[df_hourly["date"] <= now].iloc[-1]

    # ── Données journalières ──
    daily = r.Daily()
    dates_d = pd.date_range(
        start=pd.to_datetime(daily.Time(),    unit="s", utc=True).tz_convert("Europe/Paris"),
        end  =pd.to_datetime(daily.TimeEnd(), unit="s", utc=True).tz_convert("Europe/Paris"),
        freq =pd.Timedelta(seconds=daily.Interval()),
        inclusive="left"
    )
    df_daily = pd.DataFrame({
        "date":          dates_d,
        "temp_max":      daily.Variables(0).ValuesAsNumpy(),
        "temp_min":      daily.Variables(1).ValuesAsNumpy(),
        "pluie_mm":      daily.Variables(2).ValuesAsNumpy(),
        "weather_code":  daily.Variables(3).ValuesAsNumpy(),
        "vent_max_kmh":  daily.Variables(4).ValuesAsNumpy(),
    })
    df_daily["meteo"]    = df_daily["weather_code"].astype(int).map(WMO_CODES).fillna("?")
    df_daily["date_str"] = df_daily["date"].dt.strftime("%A %d/%m")

    return {
        "maintenant": {
            "temperature_c":    round(float(current["temperature_2m"]), 1),
            "ressentie_c":      round(float(current["temperature_ressentie"]), 1),
            "humidite_pct":     round(float(current["humidite_pct"])),
            "vent_kmh":         round(float(current["vent_kmh"]), 1),
            "precipitation_mm": round(float(current["precipitation_mm"]), 1),
            "meteo":            WMO_CODES.get(int(current["weather_code"]), "?"),
        },
        "previsions": df_daily[["date_str","temp_max","temp_min","pluie_mm","vent_max_kmh","meteo"]].to_dict("records"),
        "df_hourly":  df_hourly,
        "df_daily":   df_daily,
    }

print("✅ Fonction get_meteo_complet() prête")

✅ Fonction get_meteo_complet() prête


In [7]:
meteo = get_meteo_complet(43.6119, 3.8772)  # Montpellier

m = meteo["maintenant"]
print(f"📍 Montpellier — maintenant")
print(f"   {m['meteo']}")
print(f"   🌡️  {m['temperature_c']}°C  (ressenti {m['ressentie_c']}°C)")
print(f"   💨  {m['vent_kmh']} km/h")
print(f"   💧  {m['humidite_pct']}%")
print(f"   🌧️  {m['precipitation_mm']} mm\n")

print("📅 Prévisions 3 jours :")
for j in meteo["previsions"]:
    print(f"   {j['date_str']:<20} {j['meteo']:<30} {j['temp_min']:.0f}→{j['temp_max']:.0f}°C  🌧️ {j['pluie_mm']:.1f}mm  💨 {j['vent_max_kmh']:.0f}km/h")

📍 Montpellier — maintenant
   ⛅ Partiellement nuageux
   🌡️  23.8°C  (ressenti 24.9°C)
   💨  7.1 km/h
   💧  45%
   🌧️  0.0 mm

📅 Prévisions 3 jours :
   Monday 20/04         ☁️ Couvert                     14→24°C  🌧️ 0.0mm  💨 13km/h
   Tuesday 21/04        🌤️ Peu nuageux                 12→25°C  🌧️ 0.0mm  💨 11km/h
   Wednesday 22/04      🌦️ Averses légères             13→18°C  🌧️ 1.8mm  💨 16km/h


In [8]:
resp = requests.get(
    f"{DT_BASE}/catalog",
    headers=DT_HEADERS,
    params={"lang": "fr", "geo_distance": "43.6119,3.8772,10km", "page_size": 5}
)

print(f"{'Lieu':<38} {'Commune':<22} {'Météo':<25} {'°C':>6} {'Vent':>9} {'Pluie/j':>8}")
print("─" * 115)

for p in resp.json()["objects"]:
    nom   = p.get("label", {}).get("@fr", "?")[:36]
    ville = extraire_ville(p)
    geo   = p.get("isLocatedAt", [{}])[0].get("geo", {})
    lat   = float(geo.get("latitude",  0))
    lon   = float(geo.get("longitude", 0))

    m = get_meteo_complet(lat, lon)["maintenant"]

    print(f"{nom:<38} {ville:<22} {m['meteo']:<25} "
          f"{m['temperature_c']:>4}°C "
          f"{m['vent_kmh']:>6}km/h "
          f"{m['precipitation_mm']:>5}mm")

Lieu                                   Commune                Météo                         °C      Vent  Pluie/j
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
POINT D'EAU POTABLE                    Pignan                 ⛅ Partiellement nuageux   23.4°C   11.8km/h   0.0mm
CANDLELIGHT: JEAN-JACQUES GOLDMAN      Montpellier            ⛅ Partiellement nuageux   23.8°C    7.1km/h   0.0mm
L'AMBASSADOR                           Montpellier            🌤️ Peu nuageux            23.9°C    8.7km/h   0.0mm
KWM KITESCHOOL CLUB                    Villeneuve-lès-Maguelone ☀️ Ciel dégagé            21.7°C   13.4km/h   0.0mm
LE PATIO 34 - TERRASSE SUR PISCINE     Lavérune               🌤️ Peu nuageux            23.6°C   10.4km/h   0.0mm
